# Dataset integrado de precios y productos para preparación predictiva

Grupo 2. M1721, ESPOCH. Versión académica v1. El Notebook histórico permanece intacto. Esta versión trabaja con proyecciones redistribuibles y conserva la diferencia entre reproducción pública y archivo privado.

## Problema, finalidad y alcance

Los precios, características del producto y evidencias están distribuidos entre fuentes heterogéneas. La finalidad posterior es una regresión del precio. El alcance actual termina en construir y documentar el dataset; no en entrenar o evaluar modelos.

Construir un dataset integrado, trazable y reproducible a partir de Open Prices y Open Food Facts, que reúna información sobre precios, características de los productos, ubicación y momento del registro, y que quede preparado como base para el posterior desarrollo de modelos predictivos del precio de productos alimenticios.

In [1]:
from pathlib import Path
import sys, json, hashlib
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
if not (ROOT / 'data/manifests/version_v1.json').is_file():
    ROOT = ROOT.parent
assert (ROOT / 'data/manifests/version_v1.json').is_file(), 'Ejecutar desde la raíz o notebooks/'
sys.path.insert(0, str(ROOT / 'src'))
import reproducir as pipeline
pipeline.block_network()
version = json.loads((ROOT / 'data/manifests/version_v1.json').read_text(encoding='utf-8'))
display(pd.DataFrame([{'Python':sys.version.split()[0], 'pandas':pd.__version__, 'modo':'OFFLINE_DESDE_PROYECCIONES'}]))

,Python,pandas,modo
0,3.12.14,3.0.1,OFFLINE_DESDE_PROYECCIONES


## Fuentes, adquisición, formatos y metadatos

El archivo privado conserva el Parquet OP fijado, 86 respuestas OFF completas y 83 metadatos de proofs. Se verificaron sus hashes antes de proyectar. No se adquirieron nuevas fuentes. La copia pública contiene proyecciones CSV y el resultado CSV/Parquet; no los originales crudos.

El manifiesto permite identificar fuente, adquisición, revisión y hash. No debe confundirse la fecha de adquisición con la fecha del precio.

In [2]:
sources = pd.read_csv(ROOT/'data/manifests/fuentes_v1.csv', dtype='string')
display(sources[['ruta_original','adquirido_utc','snapshot','sha256']].head(6))
display(pd.DataFrame([{'originales_identificados':len(sources),'verificacion_original_publica':'No: originales excluidos por privacidad'}]))

,ruta_original,adquirido_utc,snapshot,sha256
0,data/source_archive/open_food_facts/00204682.json,2026-09-06T22:55:16.858176+00:00,<NA>,c64b3fc5e4c20ce3b513d48a0f725676c218dd586bcb90...
1,data/source_archive/open_food_facts/0021130306...,2026-09-06T22:55:22.090295+00:00,<NA>,029e122e3be26b29601c3487f8bb4600690da9766b096e...
2,data/source_archive/open_food_facts/0021130494...,2026-09-06T22:55:27.278151+00:00,<NA>,86e7f3a3babcfb308826d3e0ec333046f5adc347ac8312...
3,data/source_archive/open_food_facts/0021500042...,2026-09-06T22:55:32.644208+00:00,<NA>,ebbae8c996be59f3d148b94f0d6212e5302a8331f89131...
4,data/source_archive/open_food_facts/0034500420...,2026-09-06T22:55:37.827808+00:00,<NA>,9ff48e577c797b812aa10774c39d0afb3747737e32375d...
5,data/source_archive/open_food_facts/0048500205...,2026-09-06T22:55:43.031158+00:00,<NA>,dd7c81fdc1297c5639c630a74444ec2ddd2ea16c4cd28a...


,originales_identificados,verificacion_original_publica
0,170,No: originales excluidos por privacidad


Los hashes identifican archivos; no prueban exactitud del contenido ni disponibilidad histórica del atributo. Los originales fueron preservados antes de transformar. La construcción privada produjo proyecciones explícitas y no se presenta esa selección como adquisición original pública.

## Arquitectura SQL, NoSQL y archivos

Open Prices es el núcleo estructurado. PostgreSQL organiza productos, categorías, precios, ubicaciones y proofs mediante claves. MongoDB conserva fichas OFF como documentos completos con objetos y listas. Los proofs son entidades de evidencia; no todos son fotografías. Las imágenes seleccionadas permanecen privadas. Python hace la integración, no un JOIN entre motores.

In [3]:
pg = json.loads((ROOT/'outputs/postgresql_execution.json').read_text(encoding='utf-8'))
mongo = json.loads((ROOT/'outputs/mongodb_execution.json').read_text(encoding='utf-8'))
display(pd.DataFrame(pg['checks']))
display(pd.DataFrame([{'motor':'MongoDB','version':mongo['version'],'documentos':mongo['documents'],'consultas':mongo['queries'],'indices':len(mongo['indexes'])}]))

,check,value,status
0,rows_stg_open_prices,307183,PASS
1,rows_productos,135700,PASS
2,rows_categorias,593,PASS
3,rows_precios,307183,PASS
4,rows_ubicaciones,6608,PASS
5,rows_proofs,113917,PASS
6,view_cardinality,307183,PASS
7,proof_nulls_preserved,129,PASS


,motor,version,documentos,consultas,indices
0,MongoDB,8.0.29,86,8,4


Estos resultados son evidencia histórica de ejecución, no consultas actuales. PostgreSQL verificó una vista de 307.183 observaciones y MongoDB conservó 86 respuestas. Los scripts SQL y recursos MongoDB se incluyen para explicar el diseño histórico; requieren insumos privados y no se ejecutan aquí.

## Integración y cardinalidades

En la ejecución privada se seleccionaron del core todas las observaciones PRODUCT cuyo código coincide exactamente con una ficha OFF válida. No se filtró por moneda, ciudad, nutrición o proof. Aquí reconstruimos la integración desde esas proyecciones, comprobando unicidad de cada clave antes de unir.

In [4]:
core = pd.read_csv(ROOT/'data/inputs/core_enriquecible_v1.csv', dtype='string')
off = pd.read_csv(ROOT/'data/inputs/off_proyeccion_v1.csv', dtype='string')
proof = pd.read_csv(ROOT/'data/inputs/proofs_proyeccion_v1.csv', dtype='string')
assert core.price_id.is_unique and off.product_code.is_unique and proof.proof_id.is_unique
a = core.merge(off, on='product_code', how='left', validate='many_to_one')
b = a.merge(proof, on='proof_id', how='left', validate='many_to_one')
assert len(core) == len(a) == len(b) and b.price_id.is_unique
display(pd.DataFrame({'etapa':['selección core','unión OFF','unión proof'],'filas':[len(core),len(a),len(b)]}))

,etapa,filas
0,selección core,485
1,unión OFF,485
2,unión proof,485


Las uniones conservan 485 observaciones. La ausencia de metadatos secundarios no multiplica ni elimina precios. Los siete productos OFF no encontrados y las observaciones CATEGORY permanecen en los datasets históricos; el nuevo alcance exige una ficha válida, no modifica esos históricos.

In [5]:
d = pipeline.build(ROOT)
assert d.price_id.is_unique and d.price_type.eq('PRODUCT').all()
display(d[['price_id','product_code','price','currency','date','country','city','product_name']].head(8))

,price_id,product_code,price,currency,date,country,city,product_name
0,355,3423182102411,4.150,EUR,2023-11-27,France,Grenoble,pur jus pomme framboise bio
1,655,3760020507916,3.070,EUR,2023-11-27,France,Grenoble,Gaufrettes chocolat
2,999,3443431503004,6.350,EUR,2023-12-18,France,Paris,Poudre de cacao maigre biologique
3,1000,3456300014556,2.850,EUR,2023-12-19,France,Grenoble,Fourrés Chocolat🍫
4,1001,3760020507916,3.160,EUR,2023-12-19,France,Grenoble,Gaufrettes chocolat
5,1096,3456300014556,2.620,EUR,2023-12-24,France,Grenoble,Fourrés Chocolat🍫
6,1108,3456300014556,2.850,EUR,2023-12-24,France,Grenoble,Fourrés Chocolat🍫
7,1109,3760020507916,3.160,EUR,2023-12-24,France,Grenoble,Gaufrettes chocolat


## Resultado final del proyecto

El resultado nuevo es precios_productos_predictive_preparation_v1. El core conserva el snapshot completo; el multifuente histórico demuestra integración; v1 materializa todas las filas actualmente enriquecibles con OFF archivado y deriva componentes temporales. Ninguno se presenta como prueba de suficiencia predictiva.

In [6]:
display(pd.DataFrame([
    {'dataset':'open_prices_core_curated','filas':307183,'columnas':24,'rol':'Histórico completo, privado e intacto'},
    {'dataset':'precios_productos_multifuente_curated','filas':100,'columnas':36,'rol':'Demostración histórica, privada e intacta'},
    {'dataset':pipeline.NAME,'filas':len(d),'columnas':len(d.columns),'rol':'Preparación v1 incluida'}]))
display(pd.DataFrame({'variable':d.columns,'presentes':d.notna().sum().values,'tipo':d.dtypes.astype(str).values}))

,dataset,filas,columnas,rol
0,open_prices_core_curated,307183,24,"Histórico completo, privado e intacto"
1,precios_productos_multifuente_curated,100,36,"Demostración histórica, privada e intacta"
2,precios_productos_predictive_preparation_v1,485,38,Preparación v1 incluida


,variable,presentes,tipo
0,price_id,485,Int64
1,price_type,485,string
2,product_code,485,string
3,category_tag,0,string
4,price,485,object
5,price_per,0,string
6,currency,485,string
7,date,485,string
8,location_id,485,Int64
9,proof_id,485,Int64


## Calidad, faltantes y evidencia

Los flags históricos se conservan sin modificar su definición. qc_core_eligible incluye proof y depende del precio: no es una regla automática para seleccionar datos predictivos. Los nulos de metadatos indican información no archivada, no ausencia de evidencia en origen.

In [7]:
display(pd.DataFrame([{'flag':c,'false':int((~d[c]).sum())} for c in d if c.startswith('qc_')]))
display(pd.DataFrame([{'metadatos_proof_presentes':int(d.proof_metadata_available.sum()),'sin_metadatos_archivados':int((~d.proof_metadata_available).sum()),'filas_conservadas':len(d)}]))
display(d.nutriscore_grade.value_counts(dropna=False).rename_axis('estado').to_frame('filas'))

,flag,false
0,qc_price_positive,0
1,qc_currency_present,0
2,qc_date_present,0
3,qc_date_not_future,0
4,qc_date_valid,0
5,qc_proof_present,0
6,qc_proof_visual,410
7,qc_product_identity,0
8,qc_core_eligible,0


,metadatos_proof_presentes,sin_metadatos_archivados,filas_conservadas
0,75,410,485


,filas
estado,
d,152
c,103
e,94
a,54
unknown,52
b,29
not-applicable,1


Las 410 filas sin metadatos archivados permanecen. Un estado unknown o not-applicable de Nutri-Score no es un grado a-e. No se imputaron faltantes ni se eliminaron observaciones secundariamente.

## Preparación del dataset para una futura predicción de precios

price sería el objetivo potencial. Marca, categorías OFF, atributos seleccionables de producto, lugar y fecha son candidatos condicionados. Los identificadores se conservan para trazabilidad y no se tratan como números continuos. La matriz diferencia candidatos, descriptores, identificadores, riesgos de leakage y variables que necesitan preparación.

In [8]:
matrix = pd.read_csv(ROOT/'docs/MATRIZ_VARIABLES_PARA_FUTURO_MODELADO.csv')
display(matrix[['VARIABLE','COBERTURA','CLASIFICACION','PREPARACION_NECESARIA']])

,VARIABLE,COBERTURA,CLASIFICACION,PREPARACION_NECESARIA
0,price_id,485/485 filas,B_IDENTIFICADOR_LINAJE,No codificar como magnitud
1,price_type,485/485 filas,C_DESCRIPTIVA,Separar PRODUCT de CATEGORY
2,product_code,485/485 filas,B_IDENTIFICADOR_LINAJE,Conservar ceros; no tratar como número continuo
3,category_tag,0/485 filas,E_REQUIERE_PREPARACION,No sustituir categories_tags OFF
4,price,485/485 filas,OBJETIVO_POTENCIAL,"Delimitar moneda, cantidad y envase"
5,price_per,0/485 filas,E_REQUIERE_PREPARACION,No imputar unidad; ausente en PRODUCT
6,currency,485/485 filas,A_CANDIDATO_CON_RESTRICCION,Una moneda o estrategia posterior justificada
7,date,485/485 filas,A_PREDICTOR_CANDIDATO,Conservar original y validar formato
8,location_id,485/485 filas,B_IDENTIFICADOR_LINAJE,No magnitud continua; prever lugares nuevos
9,proof_id,485/485 filas,B_IDENTIFICADOR_LINAJE,Nulo es no consultado; False no implica inexis...


## Variables temporales y disponibilidad histórica

year, month y day_of_week derivan solo de date; lunes es 0. No se añadió quarter redundante. Los metadatos de captura OFF son distintos de la fecha del precio. Marca e identidad relativamente estables siguen sin certificación histórica; categorías, ingredientes, nutrición y clasificaciones pueden cambiar.

In [9]:
dates = pd.to_datetime(d.date)
assert d.year.eq(dates.dt.year).all()
assert d.month.eq(dates.dt.month).all()
assert d.day_of_week.eq(dates.dt.dayofweek).all()
display(d[['date','year','month','day_of_week','off_acquired_at','off_history_status']].head())

,date,year,month,day_of_week,off_acquired_at,off_history_status
0,2023-11-27,2023,11,0,2026-09-06T22:58:21.457654+00:00,NO_VERIFICABLE_HISTORICAMENTE
1,2023-11-27,2023,11,0,2026-09-05T19:36:23.581606+00:00,NO_VERIFICABLE_HISTORICAMENTE
2,2023-12-18,2023,12,0,2026-09-06T22:58:26.582736+00:00,NO_VERIFICABLE_HISTORICAMENTE
3,2023-12-19,2023,12,1,2026-09-06T22:58:37.175429+00:00,NO_VERIFICABLE_HISTORICAMENTE
4,2023-12-19,2023,12,1,2026-09-05T19:36:23.581606+00:00,NO_VERIFICABLE_HISTORICAMENTE


## Moneda, ubicación y comparabilidad

Se conserva currency. Una futura tarea deberá trabajar dentro de una moneda o justificar otra estrategia posteriormente. No hubo conversión. price_per está ausente en PRODUCT; falta resolver cantidad/envase. location_id es categórico; país y ciudad no representan coordenadas ni una tienda identificada por nombre.

In [10]:
display(d.groupby('currency').agg(filas=('price_id','size'),productos=('product_code','nunique')))
display(pd.DataFrame([{'price_per_presentes':int(d.price_per.notna().sum()),'paises':d.country.nunique(),'ubicaciones':d.location_id.nunique(),'pais_nulo':int(d.country.isna().sum()),'ciudad_nula':int(d.city.isna().sum())}]))

,filas,productos
currency,,
ADP,1,1
AED,2,1
AFN,1,1
AMD,1,1
ARS,1,1
EUR,449,66
GBP,4,1
INR,2,1
THB,1,1


,price_per_presentes,paises,ubicaciones,pais_nulo,ciudad_nula
0,0,14,148,1,6


## Información nutricional

Se revisaron 79 fichas: 77 contienen nutrition. La estructura distingue bases y preparación e incluye valores calculados. No se añadieron nutrientes numéricos, ingredientes libres ni tamaños de envase en v1. Esto evita afirmar comparabilidad que no se ha verificado; su presencia archivada queda documentada para un estudio posterior.

In [11]:
nutrition = pd.read_csv(ROOT/'outputs/nutricion_cobertura_v1.csv', dtype={'product_code':'string'})
display(nutrition.groupby(['per','preparation'],dropna=False).size().to_frame('productos'))
display(nutrition.filter(regex='presente$').sum().to_frame('productos_con_campo'))

productos
per   preparation           
100g  as_sold             73
100ml as_sold              3
      prepared             1
NaN   NaN                  2

,productos_con_campo
nutrition_presente,77
energy-kcal_presente,76
fat_presente,75
sugars_presente,75
proteins_presente,75
salt_presente,75


## Leakage y decisiones de validación futura

Se excluyen de predictores price_id, hashes, rutas, flags derivados del precio y contenido de proofs que incorpore el objetivo. No se calculan medias de precio ni agregaciones con observaciones futuras. OFF no acredita disponibilidad histórica de sus atributos.

Un estudio posterior deberá considerar dependencia por producto, ubicación, fecha y proof, así como productos o lugares nuevos. No se supone que un split aleatorio sea válido. Aquí no se divide el dataset ni se entrena un modelo.

## Exploración descriptiva conservada

Se mantiene el ejemplo de Gaufrettes chocolat, código 3760020507916, en EUR. Su función es comprobar que los datos permiten una consulta interpretable, no presentar la descripción como objetivo final. Se comparan importes del mismo código y moneda sin afirmar equivalencia histórica de envase.

In [12]:
example = d[d.product_code.eq('3760020507916') & d.currency.eq('EUR') & d.qc_price_positive & d.qc_date_valid].copy()
assert len(example) == 32
display(pd.DataFrame([{'observaciones':len(example),'fechas':example.date.nunique(),'ubicaciones':example.location_id.nunique(),'min_EUR':example.price.min(),'max_EUR':example.price.max(),'rango_EUR':example.price.max()-example.price.min()}]))
display(example[['price_id','date','location_id','price']].sort_values(['date','price_id']).head(8))

,observaciones,fechas,ubicaciones,min_EUR,max_EUR,rango_EUR
0,32,27,12,2.150,3.570,1.420


,price_id,date,location_id,price
44,12494,2023-10-09,14,2.820
45,12495,2023-10-09,14,2.820
46,12496,2023-10-09,14,2.820
1,655,2023-11-27,1,3.070
4,1001,2023-12-19,3,3.160
7,1109,2023-12-24,3,3.160
8,1124,2023-12-24,12,3.150
11,1143,2023-12-26,14,2.820


El rango observado describe estos registros. No demuestra causas, inflación ni rendimiento predictivo. Las observaciones de un mismo producto no son necesariamente independientes.

## Trazabilidad y hashes

Cada observación enlaza con el hash del core histórico, su ficha OFF y, cuando existe, los metadatos archivados del proof. Las imágenes siguen fuera de esta carpeta por privacidad. Los hashes históricos se verificaron privadamente y aquí solo se identifican.

In [13]:
lineage = pd.read_csv(ROOT/'data/manifests/linaje_filas_v1.csv', dtype='string')
assert len(lineage) == len(d) and lineage.price_id.is_unique
display(lineage[lineage.price_id.eq('42629')])
display(pd.DataFrame(list(version['outputs_sha256'].items()),columns=['archivo_v1','sha256']))

,price_id,product_code,off_source_sha256,proof_id,proof_metadata_sha256,core_source_sha256
108,42629,3560070283484,24fd7006bec7ae662e98251226eb42439e6c7802223600...,12176,0348b89a3735c597bc339ed3fbdb491f81af48d53a56cc...,810cdd8a91f2858188f8b870b13949b83af65bd19b657f...


,archivo_v1,sha256
0,precios_productos_predictive_preparation_v1.csv,ddeef932f6660e19f693d422694288655f68144d515d69...
1,precios_productos_predictive_preparation_v1.pa...,ed3c5be0c37d8e63f7890669afb19a410aaec2aadfc0fd...


## Reproducción pública y privada

La reconstrucción siguiente escribe temporales reales CSV/Parquet, verifica equivalencia y compara hashes. No usa Internet ni motores. Empieza en proyecciones públicas: no se confunde con la selección privada desde el core ni con la verificación original de 259 archivos del Notebook histórico.

In [14]:
result = pipeline.reproduce(ROOT)
assert result['estado'] == 'PASS'
display(result)

{'estado': 'PASS',
 'modo': 'OFFLINE_DESDE_PROYECCIONES',
 'shape': [485, 38],
 'price_id_unicos': 485,
 'hashes': {'precios_productos_predictive_preparation_v1.csv': 'ddeef932f6660e19f693d422694288655f68144d515d6993550bb57d91427e6c',
  'precios_productos_predictive_preparation_v1.parquet': 'ed3c5be0c37d8e63f7890669afb19a410aaec2aadfc0fd08e54f287b0e47f03a'},
 'alcance': 'Reconstruye v1 desde proyecciones incluidas; no verifica originales privados excluidos.'}

## Limitaciones y conclusiones

Se obtuvo una estructura integrada de 485 observaciones y 38 variables, con linaje, derivaciones temporales y roles de variables. Los históricos se preservaron. Las 79 fichas disponibles condicionan la selección; el tamaño, la heterogeneidad monetaria, los faltantes y la falta de versiones históricas impiden prometer una modelización seria sin trabajo adicional.

Este proyecto construye y documenta el conjunto de datos necesario para una futura tarea de predicción de precios; el entrenamiento y evaluación del modelo quedan fuera del alcance de esta entrega.

No se usaron modelos, splits, nuevas descargas, OCR ni Spark. El benchmark histórico indicaba que el volumen cabía en memoria; no se repitió. Las bases cuentan con evidencia histórica, no una ejecución nueva en este notebook. La publicación sigue pendiente de autorización.

## Referencias

[Open Prices](https://openfoodfacts.github.io/open-prices/guides/data/), [Open Food Facts API](https://openfoodfacts.github.io/openfoodfacts-server/api/) y [términos OFF](https://world.openfoodfacts.org/terms-of-use). Los manifiestos identifican URLs, hashes y fechas de las fuentes previamente archivadas. Se conserva atribución a Open Prices, Open Food Facts y sus contribuyentes. No se consultaron nuevas fuentes en esta versión.